# Tema: JSON, struct, array y datos semiestructurados

## Objetivos
Parsear JSON tipado, acceder a structs y expandir arrays manteniendo filas vacías.

## Conceptos importantes para el examen
from_json; explode/ explode_outer; tipos explícitos; quarantine; JSON string frente a struct.

**Dificultad:** Intermedio · **Tiempo estimado:** 55 min.



Ejecuta la preparación una vez; después avanza celda a celda. Las soluciones modifican datos: úsalas tras tu intento. Para volver al estado inicial, ejecuta de nuevo la preparación completa (crea otro schema). No uses «Run all» para estudiar.

- [ ] Completado
- [ ] Necesito repasar
- [ ] Dominado

## Preparación y datos ficticios
Se necesita un notebook Python en Databricks con Spark y Unity Catalog. Solo se crean objetos en el schema de prácticas mostrado.

In [ ]:
# Cada ejecución de esta celda crea un schema NUEVO y aislado.
# El catálogo debe existir y permitir USE CATALOG y CREATE SCHEMA.
# Si no puedes crear schemas, pide uno de prácticas exclusivo y cambia SCHEMA.
import re
import uuid
from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql.window import Window

dbutils.widgets.text("catalog", spark.sql("SELECT current_catalog()").first()[0])
CATALOG = dbutils.widgets.get("catalog")
RUN_ID = uuid.uuid4().hex[:10]
SCHEMA = "dea_13_" + RUN_ID
def ident(value):
    return "`" + value.replace("`", "``") + "`"
spark.sql(f"USE CATALOG {ident(CATALOG)}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {ident(SCHEMA)}")
spark.sql(f"USE SCHEMA {ident(SCHEMA)}")
spark.sql("SET TIME ZONE 'UTC'")
print(f"Objetos de esta sesión: {CATALOG}.{SCHEMA}")
# No se borran automáticamente schemas, tablas ni checkpoints.


In [ ]:
import json
payloads = [(i, json.dumps({"customer": {"id": i, "city": "Madrid" if i % 2 else "Bilbao"}, "items": [{"sku": "A", "qty": i}, {"sku": "B", "qty": 1}] if i % 4 else []})) for i in range(1, 13)]
raw = spark.createDataFrame(payloads + [(13, "{invalid")], "event_id INT, payload STRING")
raw.write.format("delta").mode("errorifexists").saveAsTable("events_raw")
JSON_SCHEMA = "customer STRUCT<id:INT,city:STRING>, items ARRAY<STRUCT<sku:STRING,qty:INT>>"

## PARTE 1 - EJEMPLOS GUIADOS

### 1. Parsear

In [ ]:
parsed = raw.withColumn("data", F.from_json("payload", JSON_SCHEMA))
display(parsed.select("event_id", "data.customer.city", "data.items"))

### 2. Expandir

In [ ]:
expanded = parsed.select("event_id", F.explode_outer("data.items").alias("item"))
display(expanded)

### 3. JSON desde SQL

In [ ]:
parsed.createOrReplaceTempView("parsed_events")
display(spark.sql("SELECT event_id, data.customer.id AS customer_id, size(data.items) AS item_count FROM parsed_events"))

## PARTE 2 - EJERCICIOS
Resuelve todos antes de abrir las soluciones. Los ejercicios se realizan en orden y pueden usar resultados anteriores.

### EJERCICIO 1
Crea una columna customer_id tipada desde el JSON.

In [ ]:
# ESCRIBE TU CÓDIGO AQUÍ


### EJERCICIO 2
Separa el evento inválido usando customer.id obligatorio; conserva payload original.

In [ ]:
# ESCRIBE TU CÓDIGO AQUÍ


### EJERCICIO 3
Cuenta cuántos eventos válidos tienen un array vacío.

In [ ]:
# ESCRIBE TU CÓDIGO AQUÍ


### EJERCICIO 4
Calcula unidades por sku sin contar filas vacías como artículos.

In [ ]:
# ESCRIBE TU CÓDIGO AQUÍ


### EJERCICIO 5
Escribe en Delta event_id, customer_id, city e items para eventos válidos. Comprueba los tipos.

In [ ]:
# ESCRIBE TU CÓDIGO AQUÍ


## PARTE 3 - PISTAS
**Pista 1:** Acceso con puntos al struct.

**Pista 2:** Un struct parcialmente nulo también puede indicar error.

**Pista 3:** size(items) = 0.

**Pista 4:** explode descarta arrays vacíos.

**Pista 5:** Un Delta admite tipos anidados.

## PARTE 4 - SOLUCIONES
**Detente aquí si todavía estás practicando.** Referencias completas para comparar después de resolver. Puedes plegar esta sección en Databricks.

### Solución 1

In [ ]:
typed = parsed.withColumn("customer_id", F.col("data.customer.id"))
display(typed)

### Solución 2

In [ ]:
quarantine = parsed.filter("data.customer.id IS NULL")
assert quarantine.count() == 1
display(quarantine)

### Solución 3

In [ ]:
empty = parsed.filter("data.customer.id IS NOT NULL AND size(data.items) = 0")
assert empty.count() == 3
display(empty)

### Solución 4

In [ ]:
items = parsed.select(F.explode("data.items").alias("item"))
display(items.groupBy("item.sku").agg(F.sum("item.qty").alias("units")))

### Solución 5

In [ ]:
silver = parsed.filter("data.customer.id IS NOT NULL").select("event_id", F.col("data.customer.id").alias("customer_id"), F.col("data.customer.city").alias("city"), F.col("data.items").alias("items"))
silver.write.format("delta").mode("overwrite").saveAsTable("events_silver")
spark.table("events_silver").printSchema()

## PARTE 5 - PREGUNTAS TIPO EXAMEN
Preguntas originales de práctica; no son preguntas oficiales.

### Pregunta 1
¿Qué convierte texto JSON en columnas anidadas tipadas?

A. to_json

B. from_json

C. split por comas siempre

D. count

### Pregunta 2
¿Qué preserva una fila con array vacío al expandir?

A. explode

B. inner join

C. explode_outer

D. DISTINCT

### Pregunta 3
¿Por qué conservar payload original?

A. Permite diagnóstico y reproceso

B. Acelera siempre todo

C. Sustituye la clave primaria

D. Impide errores de entrada

### Respuestas y explicación
**1. B** — from_json interpreta según un esquema.

**2. C** — Produce una fila con elemento nulo.

**3. A** — Permite analizar lo que llegó realmente.

## PARTE 6 - RETO FINAL
Añade un campo opcional al JSON y un array con cantidades inválidas. Construye Silver y cuarentena sin perder la relación entre evento y artículo.

Anota tu decisión, implementa el código y muestra evidencias. No se incluye solución para este reto.

In [ ]:
# TU RETO: código y verificaciones
